# V07 - Ingestao idempotente com SDK

**Objetivo:** aplicar o mesmo `NodeApply` duas vezes em um space temporario e comprovar que a chave `space + external_id` nao duplica o node.

**Idempotencia:** reexecutar a mesma carga com a mesma chave atualiza o registro existente, em vez de criar um novo. E o que torna pipelines seguros para reprocessamento.

In [ ]:
# Carrega a configuracao local e cria o cliente CDF. / Load local configuration and create the CDF client.
import os
from getpass import getpass
from pathlib import Path

from dotenv import load_dotenv
from cognite.client import ClientConfig, CogniteClient
from cognite.client.credentials import OAuthClientCredentials, Token

env_file = Path('.env')
pkg_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / '.env.example').exists()),
    Path.cwd(),
)

if env_file.exists():
    load_dotenv(env_file)
else:
    for name in ('.env', '.env.example'):
        candidate = pkg_root / name
        if candidate.exists():
            load_dotenv(candidate)
            break

if not os.environ.get('CDF_CLUSTER'):
    os.environ['CDF_CLUSTER'] = input('CDF_CLUSTER: ').strip()
if not os.environ.get('CDF_PROJECT'):
    os.environ['CDF_PROJECT'] = input('CDF_PROJECT: ').strip()

base_url = os.environ.get('CDF_URL', '').rstrip('/') or f"https://{os.environ['CDF_CLUSTER']}.cognitedata.com"
oauth_ready = env_file.exists() and all(
    os.environ.get(key) for key in ('IDP_TOKEN_URL', 'IDP_CLIENT_ID', 'IDP_CLIENT_SECRET')
)
if oauth_ready:
    scopes = os.environ.get('IDP_SCOPES', '').replace(',', ' ').split() or [f'{base_url}/.default']
    audience = os.environ.get('IDP_AUDIENCE', '')
    credentials = OAuthClientCredentials(
        token_url=os.environ['IDP_TOKEN_URL'],
        client_id=os.environ['IDP_CLIENT_ID'],
        client_secret=os.environ['IDP_CLIENT_SECRET'],
        scopes=scopes,
        **({'audience': audience} if audience else {}),
    )
else:
    # Sem .env nesta pasta: usa Bearer Token.
    bearer_token = (os.environ.get('CDF_BEARER_TOKEN') or getpass('CDF_BEARER_TOKEN: ')).strip()
    if bearer_token.lower().startswith('bearer '):
        bearer_token = bearer_token[7:].strip()
    credentials = Token(bearer_token)

client = CogniteClient(ClientConfig(
    client_name='radix-cdf-onboarding-v07',
    base_url=base_url,
    project=os.environ['CDF_PROJECT'],
    credentials=credentials,
))

## 1. Criar space e um node base

In [ ]:
from uuid import uuid4
from cognite.client.data_classes.data_modeling import NodeApply, NodeId, SpaceApply

training_space = f'sp_ur_training_v07_{uuid4().hex[:8]}'
node_id = NodeId(training_space, 'idempotent-node')
client.data_modeling.spaces.apply(SpaceApply(space=training_space, name='UR training - V07'))
client.data_modeling.instances.apply(nodes=NodeApply(space=training_space, external_id=node_id.external_id))
print('node base criado em', training_space)

## 2. Aplicar duas vezes e comprovar idempotencia
A contagem permanece 1 apos duas aplicacoes da mesma chave.

In [ ]:
nodes = client.data_modeling.instances.list(instance_type='node', space=training_space, limit=1)
assert len(nodes) == 1
payload = nodes[0].as_apply()
first_apply = client.data_modeling.instances.apply(nodes=payload)
second_apply = client.data_modeling.instances.apply(nodes=payload)
nodes_after_apply = client.data_modeling.instances.list(instance_type='node', space=training_space, limit=2)
assert len(nodes_after_apply) == 1, 'A chave estavel deveria evitar duplicacao.'
print('nodes apos duas aplicacoes:', len(nodes_after_apply))
second_apply

## Mini-exercicio
- Gere uma pequena tabela pandas (3 linhas) e converta cada linha em `NodeApply` com `external_id` estavel; aplique duas vezes e confirme 3 nodes (nao 6).
- Mude o `external_id` de uma linha e observe a criacao de um novo node.

## 3. Limpeza idempotente

In [ ]:
assert training_space.startswith('sp_ur_training_v07_')
client.data_modeling.instances.delete(nodes=node_id)
client.data_modeling.spaces.delete(training_space)
print('space_still_exists:', client.data_modeling.spaces.retrieve(training_space) is not None)